In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Tugas5Mandiri-Henji") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/23 22:18:32 WARN Utils: Your hostname, mazedev-1701 resolves to a loopback address: 127.0.1.1; using 192.168.167.98 instead (on interface wlp2s0)
26/09/23 22:18:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/23 22:18:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/23 22:18:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# Path file pada HDFS
hdfs_path = "hdfs://localhost:9000/user/mazedev/tugas5/transaksi_tugas5.csv"

# Baca File dari HDFS
df_transaksi = spark.read.csv(
    hdfs_path, 
    header=True, 
    inferSchema=True
)

# Tambahkan kolom 'pendapatan' (unit_terjual * harga_satuan)
df_transaksi_updated = df_transaksi.withColumn(
    "pendapatan", 
    col("unit_terjual") * col("harga_satuan")
)
df_transaksi_updated.show()

+--------+--------------------+----------+------------+------------+----------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|
+--------+--------------------+----------+------------+------------+----------+
|   TRX-0|   Makanan & Minuman| Purworejo|           8|       75000|    600000|
|   TRX-1|          Elektronik|      Solo|           9|       75000|    675000|
|   TRX-2|Kesehatan & Kecan...|      Solo|           6|      100000|    600000|
|   TRX-3|             Fashion|Yogyakarta|           6|      100000|    600000|
|   TRX-4|          Elektronik|Yogyakarta|           2|       75000|    150000|
|   TRX-5|Kesehatan & Kecan...|  Magelang|           4|      100000|    400000|
|   TRX-6|        Rumah Tangga|  Magelang|           6|       25000|    150000|
|   TRX-7|          Elektronik|  Semarang|           8|       50000|    400000|
|   TRX-8|             Fashion|  Semarang|           5|       25000|    125000|
|   TRX-9|          Elektronik|Yogyakart

In [6]:
import pandas as pd
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Membuat df_target
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))
df_target.show()

+----------+--------------+----------+
|      kota|target_bulanan|pic_cabang|
+----------+--------------+----------+
|  Magelang|      45000000|      Rani|
|Yogyakarta|      60000000|      Joko|
|  Semarang|      55000000|      Sari|
|      Solo|      40000000|      Bayu|
| Purworejo|      30000000|     Fitri|
+----------+--------------+----------+



In [10]:
from pyspark.sql.functions import sum as spark_sum
# Meringkas total pendapatan per kota
ringkasan = df_transaksi_updated.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan tabel target untuk menghitung pencapaian
hasil = ringkasan.join(df_target, on="kota", how="left")
hasil = hasil.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

hasil.orderBy(col("pencapaian_persen").desc()).show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
# Mendefinisikan "window": kelompokkan berdasarkan kategori, urutkan dari pendapatan tertinggi
window_spec = Window.partitionBy("kategori").orderBy(col("pendapatan").desc())

# Menambahkan kolom peringkat SETIAP transaksi di setiap kota
df_ranked = df_transaksi_updated.withColumn("rank_dalam_kategori", row_number().over(window_spec))

# Menampilkan hanya 1 transaksi teratas di setiap kota
df_ranked.filter(col("rank_dalam_kategori") <= 1) \
    .orderBy("kategori", "rank_dalam_kategori") \
    .show(10)

+--------+--------------------+----------+------------+------------+----------+-------------------+
|order_id|            kategori|      kota|unit_terjual|harga_satuan|pendapatan|rank_dalam_kategori|
+--------+--------------------+----------+------------+------------+----------+-------------------+
|  TRX-35|          Elektronik| Purworejo|           9|      150000|   1350000|                  1|
|  TRX-92|             Fashion|      Solo|           9|      150000|   1350000|                  1|
| TRX-157|Kesehatan & Kecan...|Yogyakarta|           9|      150000|   1350000|                  1|
|  TRX-81|   Makanan & Minuman|  Semarang|           9|      150000|   1350000|                  1|
| TRX-350|        Rumah Tangga|Yogyakarta|           9|      150000|   1350000|                  1|
+--------+--------------------+----------+------------+------------+----------+-------------------+



In [15]:
df_transaksi_updated.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'target'")

Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'target'


In [22]:
hasil_sql = spark.sql('''
    SELECT t.kota, tg.pic_cabang, COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota,tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')
hasil_sql.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



*Kesimpulan*
Berdasarkan hasil pengamatan setelah dilakukan analisis data menggunakan Window Function, Kota Purworejo menunjukkan data pencapaian_persen tertinggi di angka 152.16%.
Hasil analisis menggunakan SparkSQL juga menunjukkan hal yang serupa, Kota Purworejo mendapatkan jumlah transaksi yang paling banyak yaitu 116 transaksi. Ini membuktikan bahwa cabang di Kota Purworejo memiliki kinerja yang paling baik. Untuk cabang yang perlu mendapatkan perhatian lebih dari manajemen adalah Semarang dan Magelang, hal ini ditunjukkan dengan rendahnya tingkat pencapaian_persen jika dibandingkan dengan cabang lain. Kota Semarang memiliki pencapaian terendah di angka 69,4%, lalu Magelang terendah ke-2 di angka 70,3 %. Data ini diperkuat dengan data jumlah transaksi, Kota Magelang menjadi cabang dengan jumlah transaksi paling rendah yaitu 86 transaksi disusul Semarang dengan 93 transaksi